# CEX Volume & OI vs GMX Pool Depth — Correlation Analysis

## Purpose
Understand how centralized exchange (CEX) trading activity relates to GMX V2 pool liquidity,
to answer: **are GMX pools sized appropriately for each token's real market demand?**

## Metrics
| Metric | Source | What it measures |
|--------|--------|------------------|
| **CEX Daily Volume** | Binance futures 1h OHLCV | Trading demand intensity on CEX |
| **GMX Open Interest** | On-chain `OpenInterestUpdated` events | Outstanding positions on GMX |
| **GMX Pool Depth** | On-chain `PoolAmountUpdated` events (stablecoin proxy) | LP liquidity available |
| **Utilisation Ratio** | GMX OI / Pool Depth | How stressed the pool is |

## Data Gap: CEX Open Interest
**CEX OI is NOT currently collected.** The Binance feather files only contain OHLCV (no OI).
Hyperliquid volume data is all zeros (broken). To add CEX OI, install `ccxt` and use
`fetch_open_interest_history()` — but Binance limits history to 30 days.

## Analysis Levels
1. **Per-token** — Pearson r, rolling correlation, lead-lag, Granger causality
2. **Per-category** — Large (rank 1-10) / Mid (11-30) / Small (31+) by CEX volume

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import pearsonr, spearmanr
from statsmodels.tsa.stattools import grangercausalitytests
import warnings

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────────
BINANCE_DIR = Path("/Users/avik/Work/tradingstrategy/my-strategies/user_data/data/binance/futures")
GMX_OI_DIR = Path(
    "/Users/avik/Work/tradingstrategy/gmx_historical_data/user_data/data/gmx/open_interest/arbitrum/raw"
)
GMX_POOL_DIR = Path(
    "/Users/avik/Work/tradingstrategy/gmx_historical_data/user_data/data/gmx/pool_liquidity/arbitrum/snapshots"
)

# ── Constants ──────────────────────────────────────────────────────────────────
TEMPLATE = "plotly_white"
SCALE_30 = 10**30
MA_WINDOW = 30
MIN_DAYS = 60
MAX_LAG = 14  # days for lead-lag / Granger tests
CAT_COLORS = {"Large": "#1f77b4", "Mid": "#ff7f0e", "Small": "#2ca02c"}

_STABLE_TOKENS = {
    "0xaf88d065e77c8cc2239327c5edb3a432268e5831",  # USDC (native)
    "0xff970a61a04b1ca14834a43f5de4533ebddb5cc8",  # USDC.e (bridged)
    "0xfd086bc7cd5c481dcc9c85ebe478a1c0b69fcbb9",  # USDT
    "0xda10009cbd5d07dd0cecc66161fc93d7c9000da1",  # DAI
}

In [ ]:
# ── Utility functions ─────────────────────────────────────────────────────────


def _normalize_base(base: str) -> str:
    """Strip exchange-specific multiplier prefixes.

    Binance uses ``1000BONK``, Hyperliquid uses ``KBONK``; both map to ``BONK``.
    """
    base = re.sub(r"^1000", "", base)
    base = re.sub(r"^K(?=[A-Z])", "", base)
    return base


def load_binance_volume(
    data_dir: Path, norm_to_raw: dict[str, str], bases: list[str]
) -> pd.DataFrame:
    """Load Binance 1h futures feather files → daily USD volume.

    :param data_dir: Directory containing feather files.
    :param norm_to_raw: Mapping normalised base → raw filename base.
    :param bases: Normalised base symbols to load.
    :return: DataFrame with columns date, base, usd_volume.
    """
    frames = []
    for base_norm in bases:
        raw = norm_to_raw.get(base_norm)
        if raw is None:
            continue
        f = data_dir / f"{raw}_USDT_USDT-1h-futures.feather"
        if not f.exists():
            continue
        try:
            df = pd.read_feather(f)
            df["date"] = pd.to_datetime(df["date"], utc=True).dt.normalize()
            df["usd_volume"] = df["close"] * df["volume"]
            daily = df.groupby("date")["usd_volume"].sum().reset_index()
            daily["base"] = base_norm
            frames.append(daily)
        except Exception as e:
            print(f"  Skip {f.name}: {e}")
    if not frames:
        return pd.DataFrame(columns=["date", "usd_volume", "base"])
    return pd.concat(frames, ignore_index=True)


def add_rolling_ma(df: pd.DataFrame, col: str, window: int = 30) -> pd.DataFrame:
    """Add rolling MA column grouped by base.

    :param df: DataFrame with columns base, date, and ``col``.
    :param col: Column to smooth.
    :param window: Rolling window size in days.
    :return: DataFrame with additional ``{col}_ma{window}d`` column.
    """
    df = df.sort_values(["base", "date"])
    df[f"{col}_ma{window}d"] = df.groupby("base")[col].transform(
        lambda s: s.rolling(window, min_periods=max(1, window // 2)).mean()
    )
    return df


def load_gmx_oi_all(oi_dir: Path) -> pd.DataFrame:
    """Load GMX OI for every market.

    :param oi_dir: Root directory containing per-symbol subdirectories.
    :return: DataFrame with columns date, base, oi_usd.
    """
    frames = []
    for symbol_dir in sorted(oi_dir.iterdir()):
        if not symbol_dir.is_dir():
            continue
        data_file = symbol_dir / "data.parquet"
        if not data_file.exists():
            continue
        try:
            df = pd.read_parquet(data_file)
            if df.empty:
                continue
            sym = df["symbol"].iloc[0] if "symbol" in df.columns else symbol_dir.name
            sym_clean = str(sym).split("/")[0].strip()
            if sym_clean.startswith("SWAP"):
                continue
            if "nextValueUsd" in df.columns:
                df["oi_usd"] = pd.to_numeric(df["nextValueUsd"], errors="coerce") / SCALE_30
            elif "next_value_usd" in df.columns:
                df["oi_usd"] = pd.to_numeric(df["next_value_usd"], errors="coerce") / SCALE_30
            else:
                continue
            ts_col = "block_timestamp" if "block_timestamp" in df.columns else "blockTimestamp"
            df["ts"] = pd.to_datetime(df[ts_col], unit="s", utc=True)
            df["date"] = df["ts"].dt.normalize()
            df["base"] = sym_clean
            daily = df.sort_values("ts").groupby(["date", "base"])["oi_usd"].last().reset_index()
            frames.append(daily)
        except Exception as e:
            print(f"  Skip {symbol_dir.name}: {e}")
    if not frames:
        return pd.DataFrame(columns=["date", "base", "oi_usd"])
    result = pd.concat(frames, ignore_index=True)
    return result.groupby(["date", "base"])["oi_usd"].sum().reset_index()


def load_gmx_pool_all(pool_dir: Path) -> pd.DataFrame:
    """Load GMX pool liquidity (stablecoin USD proxy) for every market.

    :param pool_dir: Root directory containing per-symbol subdirectories.
    :return: DataFrame with columns date, base, pool_usd.
    """
    frames = []
    for symbol_dir in sorted(pool_dir.iterdir()):
        if not symbol_dir.is_dir():
            continue
        data_file = symbol_dir / "daily.parquet"
        if not data_file.exists():
            continue
        try:
            df = pd.read_parquet(data_file)
            if df.empty:
                continue
            sym = df["symbol"].iloc[0] if "symbol" in df.columns else symbol_dir.name
            sym_clean = str(sym).split("/")[0].strip()
            if sym_clean.startswith("SWAP"):
                continue
            df["date"] = pd.to_datetime(df["date"], utc=True)
            df["base"] = sym_clean
            df["token_lc"] = df["token"].str.lower()
            stable = (
                df[df["token_lc"].isin(_STABLE_TOKENS)]
                .groupby(["date", "base"])["pool_tokens"]
                .sum()
                .reset_index()
                .rename(columns={"pool_tokens": "pool_usd"})
            )
            frames.append(stable)
        except Exception as e:
            print(f"  Skip {symbol_dir.name}: {e}")
    if not frames:
        return pd.DataFrame(columns=["date", "base", "pool_usd"])
    result = pd.concat(frames, ignore_index=True)
    return result.groupby(["date", "base"])["pool_usd"].sum().reset_index()

## 1. Load Data

- **Binance volume** — only CEX source with valid volume (Hyperliquid data is all zeros)
- **GMX OI** — end-of-day total per token from on-chain events
- **GMX Pool** — stablecoin leg as USD proxy for pool depth

In [ ]:
# ── Discover Binance bases ────────────────────────────────────────────────────
_bin_raw = sorted(
    {f.name.split("_USDT_USDT")[0] for f in BINANCE_DIR.glob("*_USDT_USDT-1h-futures.feather")}
)
_bin_norm_to_raw: dict[str, str] = {_normalize_base(b): b for b in _bin_raw}
_bin_bases = sorted(_bin_norm_to_raw.keys())

# ── Load all sources ──────────────────────────────────────────────────────────
print("Loading Binance futures volume...")
cex_vol = load_binance_volume(BINANCE_DIR, _bin_norm_to_raw, _bin_bases)
cex_vol = add_rolling_ma(cex_vol, "usd_volume", MA_WINDOW)
print(
    f"  {cex_vol['base'].nunique()} markets, {len(cex_vol):,} rows, {cex_vol['date'].min().date()} → {cex_vol['date'].max().date()}"
)

print("Loading GMX OI...")
gmx_oi = load_gmx_oi_all(GMX_OI_DIR)
print(f"  {gmx_oi['base'].nunique()} markets, {len(gmx_oi):,} rows")

print("Loading GMX Pool...")
gmx_pool = load_gmx_pool_all(GMX_POOL_DIR)
print(f"  {gmx_pool['base'].nunique()} markets, {len(gmx_pool):,} rows")

# Universe: tokens with Binance volume AND GMX data
gmx_bases = set(gmx_oi["base"].unique()) | set(gmx_pool["base"].unique())
UNIVERSE = sorted(set(_bin_bases) & gmx_bases)
print(f"\nUniverse (Binance ∩ GMX): {len(UNIVERSE)} tokens")

In [ ]:
# ── Build master daily DataFrame ─────────────────────────────────────────────
vol_ma_col = f"usd_volume_ma{MA_WINDOW}d"

v = cex_vol[cex_vol["base"].isin(UNIVERSE)][["date", "base", "usd_volume", vol_ma_col]].copy()
v = v.rename(columns={"usd_volume": "cex_vol", vol_ma_col: "cex_vol_ma"})

o = gmx_oi[gmx_oi["base"].isin(UNIVERSE)][["date", "base", "oi_usd"]].copy()
p = gmx_pool[gmx_pool["base"].isin(UNIVERSE)][["date", "base", "pool_usd"]].copy()

master = (
    v.set_index(["date", "base"])
    .join(o.set_index(["date", "base"]), how="outer")
    .join(p.set_index(["date", "base"]), how="outer")
    .reset_index()
    .sort_values(["base", "date"])
)

# ── Utilisation ratio: GMX OI / Pool Depth ────────────────────────────────────
master["utilisation"] = master["oi_usd"] / master["pool_usd"].replace(0, np.nan)

# ── Day-over-day changes (first differences) for stationarity ─────────────────
for col in ["cex_vol_ma", "oi_usd", "pool_usd"]:
    master[f"d_{col}"] = master.groupby("base")[col].diff()

print(
    f"Master: {master.shape[0]:,} rows × {master.shape[1]} cols, {master['base'].nunique()} tokens"
)
print(f"Date range: {master['date'].min().date()} → {master['date'].max().date()}")
master[["date", "base", "cex_vol_ma", "oi_usd", "pool_usd", "utilisation"]].dropna().head()

## 2. Token Categories

Ranked by average Binance daily volume:
- **Large** = rank 1-10 (BTC, ETH, SOL, ...)
- **Mid** = rank 11-30
- **Small** = rank 31+

In [ ]:
avg_vol = (
    cex_vol[cex_vol["base"].isin(UNIVERSE)]
    .groupby("base")["usd_volume"]
    .mean()
    .sort_values(ascending=False)
    .reset_index(name="avg_daily_vol")
)
avg_vol["rank"] = range(1, len(avg_vol) + 1)
avg_vol["category"] = pd.cut(
    avg_vol["rank"], bins=[0, 10, 30, float("inf")], labels=["Large", "Mid", "Small"]
)

cat_map = avg_vol.set_index("base")["category"].to_dict()
master["category"] = master["base"].map(cat_map)

for cat in ["Large", "Mid", "Small"]:
    tokens = avg_vol[avg_vol["category"] == cat]["base"].tolist()
    print(f"{cat:6s} ({len(tokens):2d}): {', '.join(tokens)}")

fig = px.bar(
    avg_vol,
    x="avg_daily_vol",
    y="base",
    color="category",
    color_discrete_map=CAT_COLORS,
    orientation="h",
    title="Token Categories by Avg Binance Daily Futures Volume",
    labels={"avg_daily_vol": "Avg Daily Volume (USD)", "base": ""},
    template=TEMPLATE,
    height=max(400, len(avg_vol) * 18),
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

## 3. Utilisation Analysis (GMX OI / Pool Depth)

The **utilisation ratio** shows how much of the pool is committed to backing open positions.
- < 0.3 = under-utilised (idle LP capital)
- 0.3-0.7 = healthy range
- \> 0.7 = stressed pool (risk of position caps, high fees)

In [ ]:
# ── Latest utilisation snapshot per token ─────────────────────────────────────
latest = (
    master.dropna(subset=["oi_usd", "pool_usd"])
    .sort_values("date")
    .groupby("base")
    .last()
    .reset_index()
)
latest["category"] = latest["base"].map(cat_map)
latest = latest.sort_values("utilisation", ascending=False)

fig = px.bar(
    latest,
    x="utilisation",
    y="base",
    color="category",
    color_discrete_map=CAT_COLORS,
    orientation="h",
    title="GMX Utilisation Ratio (OI / Pool Depth) — Latest Snapshot",
    labels={"utilisation": "Utilisation (OI / Pool)", "base": ""},
    template=TEMPLATE,
    height=max(400, len(latest) * 16),
)
fig.add_vline(x=0.7, line_dash="dash", line_color="red", annotation_text="Stress zone")
fig.add_vline(x=0.3, line_dash="dash", line_color="green", annotation_text="Under-utilised")
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

# Top stressed markets
stressed = latest[latest["utilisation"] > 0.7][
    ["base", "category", "oi_usd", "pool_usd", "utilisation"]
]
if not stressed.empty:
    print(f"\n⚠ {len(stressed)} markets above 70% utilisation:")
    print(stressed.to_string(index=False))
else:
    print("\nNo markets above 70% utilisation.")

In [ ]:
# ── Utilisation over time by category ─────────────────────────────────────────
util_cat = (
    master.dropna(subset=["utilisation", "category"])
    .groupby(["date", "category"])
    .agg(mean_util=("utilisation", "mean"), median_util=("utilisation", "median"))
    .reset_index()
)

fig = px.line(
    util_cat,
    x="date",
    y="median_util",
    color="category",
    color_discrete_map=CAT_COLORS,
    title="Median Utilisation Ratio Over Time by Category",
    labels={"median_util": "Median Utilisation (OI/Pool)", "date": ""},
    template=TEMPLATE,
    height=400,
)
fig.add_hline(y=0.7, line_dash="dash", line_color="red", opacity=0.5)
fig.add_hline(y=0.3, line_dash="dash", line_color="green", opacity=0.5)
fig.show()

## 4. Per-Token Correlation: CEX Volume vs GMX Metrics

Two approaches:
1. **Levels** — Pearson r on raw values (captures co-movement in trends)
2. **First differences** — Pearson r on day-over-day changes (removes shared trends, tests true responsiveness)

In [ ]:
def per_token_corr(master: pd.DataFrame, min_days: int = 60) -> pd.DataFrame:
    """Compute per-token correlations between CEX volume and GMX metrics.

    Computes both level-based and first-difference correlations.

    :param master: Master DataFrame.
    :param min_days: Minimum overlapping days.
    :return: DataFrame with correlation results per token.
    """
    rows = []
    for base, g in master.groupby("base"):
        row = {"base": base, "category": g["category"].iloc[0]}

        # Levels: cex_vol_ma vs oi_usd / pool_usd
        for metric, suffix in [("oi_usd", "oi"), ("pool_usd", "pool")]:
            sub = g.dropna(subset=["cex_vol_ma", metric])
            if len(sub) >= min_days:
                r, p = pearsonr(sub["cex_vol_ma"], sub[metric])
                row[f"r_{suffix}"] = r
                row[f"p_{suffix}"] = p
            else:
                row[f"r_{suffix}"] = np.nan
                row[f"p_{suffix}"] = np.nan
            row[f"n_{suffix}"] = len(sub)

        # First differences: d_cex_vol_ma vs d_oi_usd / d_pool_usd
        for metric, suffix in [("d_oi_usd", "d_oi"), ("d_pool_usd", "d_pool")]:
            sub = g.dropna(subset=["d_cex_vol_ma", metric])
            if len(sub) >= min_days:
                r, p = pearsonr(sub["d_cex_vol_ma"], sub[metric])
                row[f"r_{suffix}"] = r
                row[f"p_{suffix}"] = p
            else:
                row[f"r_{suffix}"] = np.nan
                row[f"p_{suffix}"] = np.nan

        # Avg utilisation
        row["avg_util"] = g["utilisation"].mean()

        rows.append(row)

    return pd.DataFrame(rows).sort_values("r_oi", ascending=False, na_position="last")


corr = per_token_corr(master, MIN_DAYS)
n_valid = corr["r_oi"].notna().sum()
print(f"Tokens with valid correlation: {n_valid} / {len(corr)}")
corr[["base", "category", "r_oi", "r_pool", "r_d_oi", "r_d_pool", "avg_util"]].head(15)

In [ ]:
# ── Correlation bar charts (levels vs first-diff side by side) ─────────────────
valid = corr.dropna(subset=["r_oi"])

fig = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    subplot_titles=["Levels: CEX Vol vs GMX OI", "First-Diff: ΔCEX Vol vs ΔOI"],
    horizontal_spacing=0.05,
)
for i, (col, title) in enumerate([("r_oi", "Levels"), ("r_d_oi", "First-Diff")], 1):
    df_sorted = valid.sort_values(col, ascending=True)
    for cat in ["Large", "Mid", "Small"]:
        sub = df_sorted[df_sorted["category"] == cat]
        fig.add_trace(
            go.Bar(
                x=sub[col],
                y=sub["base"],
                orientation="h",
                name=cat if i == 1 else None,
                legendgroup=cat,
                marker_color=CAT_COLORS[cat],
                showlegend=(i == 1),
            ),
            row=1,
            col=i,
        )

fig.update_layout(
    title="Per-Token Correlation: CEX Volume ↔ GMX Open Interest",
    template=TEMPLATE,
    height=max(500, len(valid) * 16),
    barmode="overlay",
)
fig.update_xaxes(range=[-1, 1], row=1, col=1)
fig.update_xaxes(range=[-1, 1], row=1, col=2)
fig.show()

# Same for pool
valid_pool = corr.dropna(subset=["r_pool"])
fig = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    subplot_titles=["Levels: CEX Vol vs Pool", "First-Diff: ΔCEX Vol vs ΔPool"],
    horizontal_spacing=0.05,
)
for i, col in enumerate(["r_pool", "r_d_pool"], 1):
    df_sorted = valid_pool.sort_values(col, ascending=True)
    for cat in ["Large", "Mid", "Small"]:
        sub = df_sorted[df_sorted["category"] == cat]
        fig.add_trace(
            go.Bar(
                x=sub[col],
                y=sub["base"],
                orientation="h",
                name=cat if i == 1 else None,
                legendgroup=cat,
                marker_color=CAT_COLORS[cat],
                showlegend=(i == 1),
            ),
            row=1,
            col=i,
        )
fig.update_layout(
    title="Per-Token Correlation: CEX Volume ↔ GMX Pool Depth",
    template=TEMPLATE,
    height=max(500, len(valid_pool) * 16),
    barmode="overlay",
)
fig.update_xaxes(range=[-1, 1], row=1, col=1)
fig.update_xaxes(range=[-1, 1], row=1, col=2)
fig.show()

In [ ]:
# ── Top 6: time series + rolling correlation ─────────────────────────────────
top6 = corr.dropna(subset=["r_oi"]).nlargest(6, "r_oi")

for _, row in top6.iterrows():
    base, r_val = row["base"], row["r_oi"]
    sub = master[master["base"] == base].dropna(subset=["cex_vol_ma", "oi_usd"]).copy()
    sub["roll_r"] = (
        sub["cex_vol_ma"].rolling(MA_WINDOW, min_periods=MA_WINDOW // 2).corr(sub["oi_usd"])
    )

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        row_heights=[0.7, 0.3],
        subplot_titles=[f"{base} — CEX Vol vs GMX OI", f"{MA_WINDOW}d Rolling r"],
        specs=[[{"secondary_y": True}], [{}]],
    )
    fig.add_trace(
        go.Scatter(
            x=sub["date"],
            y=sub["cex_vol_ma"],
            name="CEX Vol MA",
            line=dict(color="#1f77b4", width=2),
        ),
        row=1,
        col=1,
        secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(
            x=sub["date"], y=sub["oi_usd"], name="GMX OI", line=dict(color="#d62728", width=2)
        ),
        row=1,
        col=1,
        secondary_y=True,
    )
    fig.add_trace(
        go.Scatter(
            x=sub["date"], y=sub["roll_r"], name="Rolling r", line=dict(color="#9467bd", width=1.5)
        ),
        row=2,
        col=1,
    )
    fig.add_hline(y=0, row=2, col=1, line_dash="dash", line_color="grey", opacity=0.5)
    fig.update_layout(
        title=f"{base} (r={r_val:.3f}, {row['category']})",
        template=TEMPLATE,
        height=480,
        hovermode="x unified",
    )
    fig.update_yaxes(title_text="CEX Vol (USD/day)", row=1, secondary_y=False)
    fig.update_yaxes(title_text="GMX OI (USD)", row=1, secondary_y=True)
    fig.update_yaxes(range=[-1, 1], row=2)
    fig.show()

## 5. Lead-Lag Analysis: Does CEX Volume Lead GMX?

Cross-correlation at different lags (1-14 days) to test if CEX activity **predicts** GMX changes.
Also runs Granger causality tests.

In [ ]:
def lead_lag_corr(
    master: pd.DataFrame, bases: list[str], max_lag: int = 14, min_days: int = 60
) -> pd.DataFrame:
    """Compute cross-correlation at different lags for CEX vol → GMX OI.

    Positive lag = CEX leads (CEX vol today correlated with GMX OI in N days).

    :param master: Master DataFrame.
    :param bases: Tokens to analyse.
    :param max_lag: Maximum lag in days.
    :param min_days: Minimum data points.
    :return: DataFrame with base, lag, r, p columns.
    """
    rows = []
    for base in bases:
        g = master[master["base"] == base].dropna(subset=["cex_vol_ma", "oi_usd"]).copy()
        if len(g) < min_days + max_lag:
            continue
        for lag in range(-max_lag, max_lag + 1):
            shifted = g["cex_vol_ma"].shift(lag)
            valid = g[["oi_usd"]].assign(cex_shifted=shifted).dropna()
            if len(valid) < min_days:
                continue
            r, p = pearsonr(valid["cex_shifted"], valid["oi_usd"])
            rows.append({"base": base, "lag": lag, "r": r, "p": p})
    return pd.DataFrame(rows)


# Run on top 20 tokens by volume
top20 = avg_vol.head(20)["base"].tolist()
lag_df = lead_lag_corr(master, top20, MAX_LAG, MIN_DAYS)

# Find optimal lag per token (where |r| is maximised)
optimal_lag = lag_df.loc[lag_df.groupby("base")["r"].apply(lambda x: x.abs().idxmax())]
optimal_lag = optimal_lag.merge(avg_vol[["base", "category"]], on="base")
print("Optimal lag per token (positive = CEX leads):")
print(optimal_lag[["base", "category", "lag", "r"]].sort_values("lag").to_string(index=False))

In [ ]:
# ── Lead-lag heatmap for top 20 ──────────────────────────────────────────────
if not lag_df.empty:
    pivot = lag_df.pivot(index="base", columns="lag", values="r")
    # Sort by optimal lag
    order = optimal_lag.sort_values("lag")["base"].tolist()
    pivot = pivot.reindex(order)

    fig = px.imshow(
        pivot.values,
        x=[str(c) for c in pivot.columns],
        y=pivot.index.tolist(),
        color_continuous_scale="RdYlGn",
        zmin=-1,
        zmax=1,
        title="Lead-Lag Correlation: CEX Volume → GMX OI (top 20 tokens)",
        labels={"x": "Lag (days, positive = CEX leads)", "y": "", "color": "r"},
        template=TEMPLATE,
        height=500,
        aspect="auto",
    )
    fig.show()

In [ ]:
# ── Granger Causality: does CEX volume Granger-cause GMX OI? ──────────────────
granger_results = []
for base in top20:
    g = master[master["base"] == base].dropna(subset=["cex_vol_ma", "oi_usd"]).copy()
    if len(g) < MIN_DAYS + MAX_LAG:
        continue
    try:
        # Test: CEX vol → OI (does CEX vol help predict OI?)
        data = g[["oi_usd", "cex_vol_ma"]].values
        result = grangercausalitytests(data, maxlag=7, verbose=False)
        # Take minimum p-value across lags
        min_p = min(result[lag][0]["ssr_ftest"][1] for lag in result)
        best_lag = min(result, key=lambda lag: result[lag][0]["ssr_ftest"][1])
        granger_results.append(
            {
                "base": base,
                "direction": "CEX Vol → GMX OI",
                "best_lag": best_lag,
                "min_p": min_p,
                "significant": min_p < 0.05,
            }
        )

        # Reverse: OI → CEX vol (does GMX OI help predict CEX vol?)
        data_rev = g[["cex_vol_ma", "oi_usd"]].values
        result_rev = grangercausalitytests(data_rev, maxlag=7, verbose=False)
        min_p_rev = min(result_rev[lag][0]["ssr_ftest"][1] for lag in result_rev)
        best_lag_rev = min(result_rev, key=lambda lag: result_rev[lag][0]["ssr_ftest"][1])
        granger_results.append(
            {
                "base": base,
                "direction": "GMX OI → CEX Vol",
                "best_lag": best_lag_rev,
                "min_p": min_p_rev,
                "significant": min_p_rev < 0.05,
            }
        )
    except Exception as e:
        print(f"  Granger skip {base}: {e}")

granger_df = pd.DataFrame(granger_results)
if not granger_df.empty:
    print("\nGranger Causality Results (p < 0.05 = significant):")
    pivot_g = granger_df.pivot(index="base", columns="direction", values="significant")
    pivot_g.columns = ["CEX→GMX", "GMX→CEX"]
    n_cex_leads = pivot_g["CEX→GMX"].sum()
    n_gmx_leads = pivot_g["GMX→CEX"].sum()
    print(pivot_g.to_string())
    print(f"\nCEX Granger-causes GMX: {n_cex_leads}/{len(pivot_g)} tokens")
    print(f"GMX Granger-causes CEX: {n_gmx_leads}/{len(pivot_g)} tokens")

## 6. Per-Category Aggregate

Aggregate all tokens in each category to see macro trends.

In [ ]:
cat_daily = (
    master.dropna(subset=["category"])
    .groupby(["date", "category"])
    .agg(
        cex_vol=("cex_vol_ma", "sum"),
        oi=("oi_usd", "sum"),
        pool=("pool_usd", "sum"),
        n=("base", "nunique"),
    )
    .reset_index()
)

categories = ["Large", "Mid", "Small"]
fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    subplot_titles=[f"{c} Cap" for c in categories],
    specs=[[{"secondary_y": True}]] * 3,
)
for i, cat in enumerate(categories, 1):
    s = cat_daily[cat_daily["category"] == cat].sort_values("date")
    fig.add_trace(
        go.Scatter(
            x=s["date"],
            y=s["cex_vol"],
            name=f"{cat} CEX Vol",
            line=dict(color=CAT_COLORS[cat], width=2),
        ),
        row=i,
        col=1,
        secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(
            x=s["date"],
            y=s["oi"],
            name=f"{cat} OI",
            line=dict(color="#d62728", width=1.5, dash="dot"),
        ),
        row=i,
        col=1,
        secondary_y=True,
    )
    fig.add_trace(
        go.Scatter(
            x=s["date"],
            y=s["pool"],
            name=f"{cat} Pool",
            line=dict(color="#2ca02c", width=1.5, dash="dash"),
        ),
        row=i,
        col=1,
        secondary_y=True,
    )
    fig.update_yaxes(title_text="CEX Vol", row=i, secondary_y=False)
    fig.update_yaxes(title_text="OI / Pool (USD)", row=i, secondary_y=True)

fig.update_layout(
    title="Category Aggregates Over Time", template=TEMPLATE, height=900, hovermode="x unified"
)
fig.show()

In [ ]:
# ── Category correlations (levels + first-diff) ──────────────────────────────
cat_corrs = []
for cat in categories:
    s = cat_daily[cat_daily["category"] == cat].sort_values("date")
    for metric, label in [("oi", "OI"), ("pool", "Pool")]:
        sub = s.dropna(subset=["cex_vol", metric])
        if len(sub) >= MIN_DAYS:
            r_level, _ = pearsonr(sub["cex_vol"], sub[metric])
            r_diff, _ = pearsonr(sub["cex_vol"].diff().dropna(), sub[metric].diff().dropna())
        else:
            r_level, r_diff = np.nan, np.nan
        cat_corrs.append(
            {"category": cat, "metric": label, "r_levels": r_level, "r_first_diff": r_diff}
        )

cat_corr_df = pd.DataFrame(cat_corrs)
print(cat_corr_df.to_string(index=False))

fig = px.bar(
    cat_corr_df.melt(
        id_vars=["category", "metric"],
        value_vars=["r_levels", "r_first_diff"],
        var_name="method",
        value_name="r",
    ),
    x="category",
    y="r",
    color="metric",
    facet_col="method",
    barmode="group",
    template=TEMPLATE,
    height=400,
    title="Category Correlation: Levels vs First-Difference",
    color_discrete_sequence=["#d62728", "#2ca02c"],
)
fig.update_yaxes(range=[-1, 1])
fig.show()

In [ ]:
# ── Violin: per-token correlation distribution by category ────────────────────
fig = make_subplots(rows=1, cols=2, subplot_titles=["CEX Vol vs GMX OI", "CEX Vol vs GMX Pool"])

for cat in categories:
    sub = corr[corr["category"] == cat]
    oi_vals = sub["r_oi"].dropna()
    pool_vals = sub["r_pool"].dropna()
    if not oi_vals.empty:
        fig.add_trace(
            go.Violin(
                y=oi_vals,
                name=cat,
                legendgroup=cat,
                marker_color=CAT_COLORS[cat],
                box_visible=True,
                meanline_visible=True,
                points="all",
                showlegend=True,
            ),
            row=1,
            col=1,
        )
    if not pool_vals.empty:
        fig.add_trace(
            go.Violin(
                y=pool_vals,
                name=cat,
                legendgroup=cat,
                marker_color=CAT_COLORS[cat],
                box_visible=True,
                meanline_visible=True,
                points="all",
                showlegend=False,
            ),
            row=1,
            col=2,
        )

fig.update_layout(title="Correlation Distribution by Category", template=TEMPLATE, height=500)
fig.update_yaxes(range=[-1, 1])
fig.show()

## 7. Cross-Sectional Scatter: CEX Volume vs GMX Metrics

In [ ]:
# ── Scatter: avg metrics per token (log-log) ─────────────────────────────────
avgs = (
    master.groupby("base")
    .agg(
        avg_vol=("cex_vol_ma", "mean"),
        avg_oi=("oi_usd", "mean"),
        avg_pool=("pool_usd", "mean"),
        avg_util=("utilisation", "mean"),
        category=("category", "first"),
    )
    .dropna(subset=["avg_vol"])
    .reset_index()
)

for metric, label in [("avg_oi", "Avg GMX OI (USD)"), ("avg_pool", "Avg GMX Pool (USD)")]:
    sub = avgs.dropna(subset=[metric])
    fig = px.scatter(
        sub,
        x="avg_vol",
        y=metric,
        color="category",
        color_discrete_map=CAT_COLORS,
        text="base",
        log_x=True,
        log_y=True,
        trendline="ols",
        title=f"Binance Avg Volume vs {label} (log-log)",
        labels={"avg_vol": "Avg Binance Volume (USD/day)", metric: label},
        template=TEMPLATE,
        height=550,
    )
    fig.update_traces(textposition="top center", textfont_size=8)
    fig.show()

# Volume vs utilisation
fig = px.scatter(
    avgs.dropna(subset=["avg_util"]),
    x="avg_vol",
    y="avg_util",
    color="category",
    color_discrete_map=CAT_COLORS,
    text="base",
    log_x=True,
    title="Binance Volume vs GMX Utilisation Ratio",
    labels={"avg_vol": "Avg Binance Volume (USD/day)", "avg_util": "Avg Utilisation (OI/Pool)"},
    template=TEMPLATE,
    height=500,
)
fig.update_traces(textposition="top center", textfont_size=8)
fig.add_hline(y=0.7, line_dash="dash", line_color="red", opacity=0.5)
fig.show()

## 8. Summary

In [ ]:
# ── Heatmap: all tokens ──────────────────────────────────────────────────────
hm = corr.dropna(subset=["r_oi", "r_pool"], how="all").sort_values(
    ["category", "r_oi"], ascending=[True, False]
)
labels = [f"{r['base']} [{r['category']}]" for _, r in hm.iterrows()]
z = hm[["r_oi", "r_pool", "r_d_oi", "r_d_pool"]].values

fig = go.Figure(
    go.Heatmap(
        z=z,
        x=["Vol↔OI (levels)", "Vol↔Pool (levels)", "ΔVol↔ΔOI (diff)", "ΔVol↔ΔPool (diff)"],
        y=labels,
        colorscale="RdYlGn",
        zmin=-1,
        zmax=1,
        text=np.round(z, 2),
        texttemplate="%{text}",
        textfont={"size": 8},
    )
)
fig.update_layout(
    title="Correlation Heatmap: All Tokens",
    template=TEMPLATE,
    height=max(500, len(hm) * 18),
    yaxis={"dtick": 1},
)
fig.show()

In [ ]:
# ── Statistical summary ──────────────────────────────────────────────────────
print("=" * 80)
print("STATISTICAL SUMMARY BY CATEGORY")
print("=" * 80)

for cat in categories:
    sub = corr[corr["category"] == cat]
    r_oi = sub["r_oi"].dropna()
    r_pool = sub["r_pool"].dropna()
    r_d_oi = sub["r_d_oi"].dropna()
    p_oi = sub["p_oi"].dropna()
    util = sub["avg_util"].dropna()

    print(f"\n{cat} cap ({len(sub)} tokens, {len(r_oi)} with valid correlation):")
    if not r_oi.empty:
        print(
            f"  CEX Vol ↔ OI  (levels):     median r = {r_oi.median():.3f}, mean = {r_oi.mean():.3f}, {(r_oi > 0).mean() * 100:.0f}% positive, {(p_oi < 0.05).mean() * 100:.0f}% significant"
        )
    if not r_d_oi.empty:
        print(
            f"  CEX Vol ↔ OI  (first-diff):  median r = {r_d_oi.median():.3f}, mean = {r_d_oi.mean():.3f}"
        )
    if not r_pool.empty:
        print(
            f"  CEX Vol ↔ Pool (levels):     median r = {r_pool.median():.3f}, mean = {r_pool.mean():.3f}"
        )
    if not util.empty:
        print(
            f"  Avg utilisation:             median = {util.median():.3f}, mean = {util.mean():.3f}"
        )

print(f"\n{'=' * 80}")
print("KEY FINDINGS")
print(f"{'=' * 80}")

oi_all = corr["r_oi"].dropna()
pool_all = corr["r_pool"].dropna()
d_oi_all = corr["r_d_oi"].dropna()
print(f"\nOverall (n={len(oi_all)} tokens):")
print(
    f"  CEX Vol ↔ OI:   median r = {oi_all.median():.3f} (levels), {d_oi_all.median():.3f} (first-diff)"
)
print(f"  CEX Vol ↔ Pool: median r = {pool_all.median():.3f} (levels)")

best = corr.loc[corr["r_oi"].idxmax()]
worst = corr.loc[corr["r_oi"].idxmin()]
print(f"\nStrongest +r (OI): {best['base']} = {best['r_oi']:.3f} ({best['category']})")
print(f"Strongest -r (OI): {worst['base']} = {worst['r_oi']:.3f} ({worst['category']})")

print("\nNOTE: CEX OI is NOT currently collected — only CEX volume is available.")
print("Install ccxt and use fetch_open_interest_history() to add Binance OI (30-day limit).")
print(f"{'=' * 80}")

In [ ]:
# ── Export ────────────────────────────────────────────────────────────────────
out = Path("../data/cex_gmx_correlations.csv")
out.parent.mkdir(parents=True, exist_ok=True)
corr.to_csv(out, index=False)
print(f"Saved {len(corr)} rows → {out.resolve()}")